In [2]:
import numpy as np
import pandas as pd
from scipy.interpolate import PchipInterpolator

## Load SW-All source file

Single source for both F10.7_OBS (daily) and KP1-KP8 (3-hourly).

In [3]:
sw = pd.read_csv('../drivers/SW-All_20260131.csv')
sw['datetime'] = pd.to_datetime(sw['DATE'].astype(str), format='%Y-%m-%d')
print(sw.shape, sw.datetime.min(), sw.datetime.max())
sw.head()

(25191, 32) 1957-10-01 00:00:00 2041-10-01 00:00:00


,DATE,BSRN,ND,KP1,KP2,KP3,KP4,KP5,KP6,KP7,...,C9,ISN,F10.7_OBS,F10.7_ADJ,F10.7_DATA_TYPE,F10.7_OBS_CENTER81,F10.7_OBS_LAST81,F10.7_ADJ_CENTER81,F10.7_ADJ_LAST81,datetime
0,1957-10-01,1700,19,43.0,40.0,30.0,20.0,37.0,23.0,43.0,...,5.0,334,269.3,269.8,OBS,266.6,230.9,266.8,235.5,1957-10-01
1,1957-10-02,1700,20,37.0,37.0,17.0,17.0,27.0,23.0,17.0,...,3.0,331,253.3,253.6,OBS,267.4,231.7,267.5,236.2,1957-10-02
2,1957-10-03,1700,21,27.0,20.0,13.0,33.0,37.0,47.0,43.0,...,5.0,343,266.3,266.4,OBS,268.1,232.7,268.1,237.1,1957-10-03
3,1957-10-04,1700,22,30.0,30.0,23.0,27.0,23.0,27.0,30.0,...,3.0,307,238.2,238.2,OBS,268.8,233.3,268.7,237.7,1957-10-04
4,1957-10-05,1700,23,30.0,30.0,17.0,23.0,20.0,27.0,27.0,...,3.0,310,246.2,246.0,OBS,269.3,233.9,269.1,238.1,1957-10-05


## Interpolate F10.7_OBS: daily → hourly (PCHIP)

In [4]:
df_f = (
    sw[['datetime', 'F10.7_OBS']]
    .rename(columns={'F10.7_OBS': 'f107_obs'})
    .sort_values('datetime')
    .drop_duplicates('datetime')
    .copy()
)
df_f['f107_obs'] = pd.to_numeric(df_f['f107_obs'], errors='coerce')

t0 = df_f['datetime'].iloc[0]
x  = (df_f['datetime'] - t0).dt.total_seconds().to_numpy() / 3600.0
y  = df_f['f107_obs'].to_numpy()

t_hourly = pd.date_range(
    df_f['datetime'].min(),
    df_f['datetime'].max() + pd.Timedelta(hours=23),
    freq='h'
)
xq = (t_hourly - t0).total_seconds().to_numpy() / 3600.0
yq = PchipInterpolator(x, y, extrapolate=False)(xq)

cel_f107_df = (
    pd.DataFrame({'datetime': t_hourly, 'f107_obs': yq})
    .dropna()
    .reset_index(drop=True)
)
print(cel_f107_df.shape)
cel_f107_df.head()

(736345, 2)


,datetime,f107_obs
0,1957-10-01 00:00:00,269.300000
1,1957-10-01 01:00:00,268.051845
2,1957-10-01 02:00:00,266.849479
3,1957-10-01 03:00:00,265.693555
4,1957-10-01 04:00:00,264.584722


## Interpolate Kp: 3-hourly → hourly (PCHIP), then ÷10

In [5]:
# Reshape 8 3-hourly Kp columns into long format
kp_long = (
    sw.set_index('datetime')
    .filter(like='KP')
    .iloc[:, :-1]  # drop KP_SUM
    .stack()
    .reset_index()
    .rename(columns={'level_1': 'kp_time', 0: 'kp'})
)

df_kp = kp_long.copy()
df_kp['kp_idx'] = df_kp['kp_time'].str.extract(r'KP(\d+)').astype(int) - 1
df_kp['t3h']    = df_kp['datetime'] + pd.to_timedelta(df_kp['kp_idx'] * 3, unit='h')
df_kp['kp']     = pd.to_numeric(df_kp['kp'], errors='coerce')
df_kp['day']    = df_kp['datetime'].dt.floor('D')

next_day_kp0 = (
    df_kp.sort_values(['day', 't3h'])
         .groupby('day')['kp'].first()
         .shift(-1)
)

out = []
for day in next_day_kp0.index[:-1]:
    g = df_kp[df_kp['day'] == day].sort_values('t3h')
    x = ((g['t3h'] - day).dt.total_seconds().to_numpy()) / 3600.0
    y = g['kp'].to_numpy(dtype=float)
    if len(x) < 2 or np.any(~np.isfinite(y)):
        continue
    y24 = float(next_day_kp0.loc[day])
    x = np.r_[x, 24.0]
    y = np.r_[y, y24]
    order = np.argsort(x)
    x, y = x[order], y[order]
    xq2 = np.arange(24.0)
    yq2 = PchipInterpolator(x, y, extrapolate=False)(xq2)
    out.append(pd.DataFrame({'datetime': day + pd.to_timedelta(xq2, unit='h'), 'kp': yq2}))

kp_hourly_smooth = pd.concat(out, ignore_index=True)
kp_hourly_smooth['kp'] /= 10.0
print(kp_hourly_smooth.shape)
kp_hourly_smooth.head()

(600072, 2)


,datetime,kp
0,1957-10-01 00:00:00,4.300000
1,1957-10-01 01:00:00,4.256410
2,1957-10-01 02:00:00,4.146154
3,1957-10-01 03:00:00,4.000000
4,1957-10-01 04:00:00,3.746439


## Merge f107 + kp and add trig/time features

In [6]:
celestrak_drivers_df = pd.merge(cel_f107_df, kp_hourly_smooth, on='datetime', how='inner')

doy_c = np.cos(2*np.pi*(celestrak_drivers_df.datetime.dt.day_of_year.values
                        + celestrak_drivers_df.datetime.dt.hour.values/24.)/365.25)
doy_s = np.sin(2*np.pi*(celestrak_drivers_df.datetime.dt.day_of_year.values
                        + celestrak_drivers_df.datetime.dt.hour.values/24.)/365.25)
ut_c  = np.cos(2*np.pi*celestrak_drivers_df.datetime.dt.hour.values/24.)
ut_s  = np.sin(2*np.pi*celestrak_drivers_df.datetime.dt.hour.values/24.)

celestrak_drivers_df['year'] = celestrak_drivers_df.datetime.dt.year
celestrak_drivers_df['doy']  = celestrak_drivers_df.datetime.dt.day_of_year
celestrak_drivers_df['hour'] = celestrak_drivers_df.datetime.dt.hour
celestrak_drivers_df['t1']   = doy_c
celestrak_drivers_df['t2']   = doy_s
celestrak_drivers_df['t3']   = ut_c
celestrak_drivers_df['t4']   = ut_s

celestrak_drivers_df = celestrak_drivers_df[
    ['year', 't1', 't2', 't3', 't4', 'f107_obs', 'kp', 'datetime', 'doy', 'hour']
]
print(celestrak_drivers_df.shape)
celestrak_drivers_df.head()

(600072, 10)


,year,t1,t2,t3,t4,f107_obs,kp,datetime,doy,hour
0,1957,0.001075,-0.999999,1.000000,0.000000,269.300000,4.300000,1957-10-01 00:00:00,274,0
1,1957,0.001792,-0.999998,0.965926,0.258819,268.051845,4.256410,1957-10-01 01:00:00,274,1
2,1957,0.002509,-0.999997,0.866025,0.500000,266.849479,4.146154,1957-10-01 02:00:00,274,2
3,1957,0.003225,-0.999995,0.707107,0.707107,265.693555,4.000000,1957-10-01 03:00:00,274,3
4,1957,0.003942,-0.999992,0.500000,0.866025,264.584722,3.746439,1957-10-01 04:00:00,274,4


## Write v2 CSV

In [6]:
celestrak_drivers_df.to_csv(
    '../drivers/celestrak_drivers_preprocessed_20260131_v2.csv', index=False
)
print('written')

written
